In [ ]:
import sys
from pathlib import Path

# Install libraries used in this notebook
!{sys.executable} -m pip install -q numpy pandas scikit-learn torch xgboost tabpfn joblib

# Resolve the repository root in a portable way
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "data" / "input").exists():
        ROOT_DIR = candidate.resolve()
        break
else:
    ROOT_DIR = Path(r"e:/LYQ/work/250916remote_model").resolve()

DATA_DIR = ROOT_DIR / "data" / "input"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT_DIR)
print("Data directory:", DATA_DIR)


Repository root: E:\LYQ\work\250916remote_model
Data directory: E:\LYQ\work\250916remote_model\data\input


# UAV

## Create GroupKFold splits and save them

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


def build_group_kfold_split(groups, n_splits=5, seed=42):
    np.random.seed(seed)

    unique_groups = np.unique(groups)
    np.random.shuffle(unique_groups)

    fold_size = len(unique_groups) // n_splits
    folds = []

    for i in range(n_splits):
        test_groups = unique_groups[i * fold_size:(i + 1) * fold_size]
        train_groups = np.setdiff1d(unique_groups, test_groups)

        train_idx = np.where(np.isin(groups, train_groups))[0].tolist()
        test_idx = np.where(np.isin(groups, test_groups))[0].tolist()

        folds.append({
            "train_idx": train_idx,
            "test_idx": test_idx
        })

    return folds


# Create GroupKFold splits from the UAV data
csv_path = DATA_DIR / "UAV.csv"
df = pd.read_csv(csv_path, encoding="gbk")

# Group rule: every 10 rows form one sample group
groups = np.arange(len(df)) // 10

folds = build_group_kfold_split(groups, n_splits=5, seed=12)

save_path = DATA_DIR / "UAV" / "uav_splits.json"
save_path.parent.mkdir(parents=True, exist_ok=True)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(folds, f, indent=2)

print("Saved split to:", save_path)

Saved split to: e:/LYQ/work/250916remote_model/data/input/UAV/uav_splits.json


## Shared loading and evaluation utilities

In [ ]:
import json
import numpy as np

def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]

    train_idx = np.array(split["train_idx"])
    test_idx = np.array(split["test_idx"])

    return train_idx, test_idx

## CSF-PE Transformer（Continuous Spectral Fusion with Positional Encoding）

In [ ]:
import copy
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, Dataset, Subset

DATA_DIR = ROOT_DIR / "data" / "input"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"
model_save_dir = DATA_DIR / "UAV" / "CSF-FPE"
model_save_dir.mkdir(parents=True, exist_ok=True)

# Dataset selection
selected_datasets = ["uav"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
}

# Spectral normalization settings
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# Shared split loader
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),(740,15),
                     (783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        else:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # fusion layer for concatenated features
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. Learnable spectral diffusion weights
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, 32),
            nn.GELU(),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

        # =========================
        # 5. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.LayerNorm(d_model)
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()


================ FOLD 0 ================
Epoch 000 | TrainLoss 1164.8164 | Test RMSE 5.3112 R2 -9.8310
Epoch 001 | TrainLoss 733.7761 | Test RMSE 3.9699 R2 -5.0512
Epoch 002 | TrainLoss 383.2418 | Test RMSE 2.4953 R2 -1.3908
Epoch 003 | TrainLoss 162.8139 | Test RMSE 1.5659 R2 0.0585
Epoch 004 | TrainLoss 74.8074 | Test RMSE 1.4065 R2 0.2405
Epoch 005 | TrainLoss 57.1582 | Test RMSE 1.5403 R2 0.0890
Epoch 006 | TrainLoss 55.9085 | Test RMSE 1.5162 R2 0.1173
Epoch 007 | TrainLoss 54.3842 | Test RMSE 1.5567 R2 0.0695
Epoch 008 | TrainLoss 53.0681 | Test RMSE 1.2882 R2 0.3628
Epoch 009 | TrainLoss 52.8175 | Test RMSE 1.3054 R2 0.3457
Epoch 010 | TrainLoss 49.3924 | Test RMSE 1.4491 R2 0.1938
Epoch 011 | TrainLoss 51.7269 | Test RMSE 1.3240 R2 0.3269
Epoch 012 | TrainLoss 48.2997 | Test RMSE 1.2876 R2 0.3634
Epoch 013 | TrainLoss 45.3937 | Test RMSE 1.3044 R2 0.3467
Epoch 014 | TrainLoss 44.3464 | Test RMSE 1.2028 R2 0.4445
Epoch 015 | TrainLoss 41.8265 | Test RMSE 1.4774 R2 0.1619
Epoch 

## TabPFN

In [ ]:
import json
import os
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tabpfn import TabPFNRegressor

# Ignore warnings for cleaner output
warnings.filterwarnings("ignore")

# Path configuration
DATA_DIR = ROOT_DIR / "data" / "input"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"
csv_path = DATA_DIR / "UAV.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load split（统一接口）
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics（统一PI风格）
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# data loading（保持你原逻辑）
# =========================
raw = pd.read_csv(csv_path, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = max(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print("samples:", len(y), "features:", X.shape[1])
print("group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# result container
# =========================
results = []

# =========================
# 5-fold training 
# =========================
for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # ===== 加标准化 =====
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =========================
    # model
    # =========================
    model = TabPFNRegressor(
        n_estimators=32,
        device=device
    )

    # =========================
    # train
    # =========================
    model.fit(X_train, y_train)

    # =========================
    # predict
    # =========================
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # =========================
    # evaluate
    # =========================
    train_metrics = evaluate(y_train, pred_train)
    test_metrics = evaluate(y_test, pred_test)

    print(f"[Train] MSE={train_metrics[0]:.4f}  RMSE={train_metrics[1]:.4f}  MAE={train_metrics[2]:.4f} R2={train_metrics[3]:.4f}")
    print(f"[Test ] MSE={test_metrics[0]:.4f}  RMSE={test_metrics[1]:.4f}  MAE={test_metrics[2]:.4f} R2={test_metrics[3]:.4f}")

    # =========================
    # save fold result
    # =========================
    results.append(test_metrics)

    # TabPFN没有可训练权重，这里保存预测结果
    save_dir = DATA_DIR / "UAV" / "TabPFN"
    save_dir.mkdir(parents=True, exist_ok=True)

    np.save(save_dir / f"fold_{fold_id}_pred.npy", pred_test)

# =========================
# final summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
samples: 570 features: 165
group samples: 57

================ FOLD 0 ================
[Train] MSE=0.0000  RMSE=0.0069  MAE=0.0053 R2=1.0000
[Test ] MSE=1.0614  RMSE=1.0303  MAE=0.6893 R2=0.5925

================ FOLD 1 ================
[Train] MSE=0.0000  RMSE=0.0062  MAE=0.0051 R2=1.0000
[Test ] MSE=2.3101  RMSE=1.5199  MAE=1.0038 R2=0.3233

================ FOLD 2 ================
[Train] MSE=0.0000  RMSE=0.0067  MAE=0.0055 R2=1.0000
[Test ] MSE=0.7289  RMSE=0.8538  MAE=0.6612 R2=0.7015

================ FOLD 3 ================
[Train] MSE=0.0000  RMSE=0.0061  MAE=0.0048 R2=1.0000
[Test ] MSE=0.6769  RMSE=0.8227  MAE=0.6013 R2=0.8040

================ FOLD 4 ================
[Train] MSE=0.0000  RMSE=0.0060  MAE=0.0049 R2=1.0000
[Test ] MSE=5.3417  RMSE=2.3112  MAE=1.3374 R2=-0.3169

================ FINAL 5-FOLD RESULT ================
MSE  : 2.0238 ± 1.7611
RMSE : 1.3076 ± 0.5604
MAE  : 0.8586 ± 0.2772
R2   : 0.4209 ± 0.4022


================ FINAL 5-FOLD RESULT ================
MSE  : 2.2671 ± 1.9561
RMSE : 1.3862 ± 0.5879
MAE  : 0.9059 ± 0.3000
R2   : 0.3528 ± 0.4469

## Transformer baseline

In [ ]:
import copy
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Path configuration
DATA_DIR = ROOT_DIR / "data" / "input"
CSV_PATH = DATA_DIR / "UAV.csv"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load split
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# Transformer baseline model
# =========================
class TransformerBaseline(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embed = nn.Linear(1, d_model)

        self.pos = nn.Parameter(torch.randn(1, 200, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)
        x = self.embed(x)

        N = x.size(1)
        x = x + self.pos[:, :N, :].to(x.device)

        x = self.encoder(x)

        x = x.mean(dim=1)

        return self.head(x).squeeze()

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = max(165, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = TransformerBaseline().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            pred = model(X_test).cpu().numpy()
        mse, rmse, mae, r2 = evaluate(y_test, pred)
        # early stopping
        if rmse < best_loss:
            best_loss = rmse
            best_model = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered")
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    # =========================
    # save
    # =========================
    save_dir = DATA_DIR / "UAV" / "Transformer"
    save_dir.mkdir(parents=True, exist_ok=True)

    np.save(save_dir / f"fold_{fold_id}_pred.npy", pred)
    torch.save(best_model, save_dir / f"fold_{fold_id}.pth")

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 165
Group samples: 57

================ FOLD 0 ================
Early stopping triggered
MSE=1.4209, RMSE=1.1920, MAE=0.8968, R2=0.4544

================ FOLD 1 ================
Early stopping triggered
MSE=1.6659, RMSE=1.2907, MAE=1.1237, R2=0.5120

================ FOLD 2 ================
Early stopping triggered
MSE=1.1467, RMSE=1.0708, MAE=0.8894, R2=0.5304

================ FOLD 3 ================
Early stopping triggered
MSE=3.4539, RMSE=1.8585, MAE=1.4512, R2=0.0001

================ FOLD 4 ================
Early stopping triggered
MSE=1.8991, RMSE=1.3781, MAE=1.0365, R2=0.5318

================ FINAL 5-FOLD RESULT ================
MSE  : 1.9173 ± 0.8081
RMSE : 1.3580 ± 0.2703
MAE  : 1.0795 ± 0.2057
R2   : 0.4057 ± 0.2048


================ FOLD 0 ================
RMSE=1.3121, MAE=1.0682, R2=0.3390

================ FOLD 1 ================
RMSE=1.4237, MAE=1.0432, R2=0.4063

================ FOLD 2 ================
RMSE=1.5202, MAE=1.1638, R2=0.0535

================ FOLD 3 ================
RMSE=1.3348, MAE=1.0155, R2=0.4842

================ FOLD 4 ================
RMSE=1.3880, MAE=1.1283, R2=0.5250

## Residual MLP

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Path configuration
DATA_DIR = ROOT_DIR / "data" / "input"
CSV_PATH = DATA_DIR / "UAV.csv"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load folds
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# MLP model
# =========================
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

        # 如果维度不同，用 projection
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.relu(self.fc(x) + self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.block1 = ResidualBlock(in_dim, 256)
        self.block2 = ResidualBlock(256, 128)
        self.block3 = ResidualBlock(128, 64)
        self.block4 = nn.Linear(64, 32)
        self.block5 = ResidualBlock(32, 16)
        self.block6 = ResidualBlock(16, 8)
        self.block7 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.block7(x)
        return x.squeeze()

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")

raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = max(165, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values.astype(np.float32)
y = data.iloc[:, -1].values.astype(np.float32)

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = MLP(X.shape[1]).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            pred = model(X_test).cpu().numpy()
        mse, rmse, mae, r2 = evaluate(y_test, pred)
        # early stopping
        if rmse < best_loss:
            best_loss = rmse
            best_model = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                # print("Early stopping triggered")
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    # =========================
    # save
    # =========================
    save_dir = DATA_DIR / "UAV" / "MLP"
    save_dir.mkdir(parents=True, exist_ok=True)

    np.save(save_dir / f"fold_{fold_id}_pred.npy", pred)
    torch.save(best_model, save_dir / f"fold_{fold_id}.pth")

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 165
Group samples: 57

================ FOLD 0 ================
MSE=0.4200, RMSE=0.6481, MAE=0.4965, R2=0.8387

================ FOLD 1 ================
MSE=3.3336, RMSE=1.8258, MAE=1.2743, R2=0.0235

================ FOLD 2 ================
MSE=3.8467, RMSE=1.9613, MAE=1.3302, R2=-0.5755

================ FOLD 3 ================
MSE=0.9160, RMSE=0.9571, MAE=0.7255, R2=0.7348

================ FOLD 4 ================
MSE=3.1586, RMSE=1.7772, MAE=1.1521, R2=0.2213

================ FINAL 5-FOLD RESULT ================
MSE  : 2.3350 ± 1.3886
RMSE : 1.4339 ± 0.5281
MAE  : 0.9957 ± 0.3275
R2   : 0.2486 ± 0.5129


================ FOLD 0 ================
RMSE=0.7503, MAE=0.5501, R2=0.7838

================ FOLD 1 ================
RMSE=1.6949, MAE=1.2513, R2=0.1586

================ FOLD 2 ================
RMSE=1.2279, MAE=0.7146, R2=0.3825

================ FOLD 3 ================
RMSE=1.6735, MAE=1.2167, R2=0.1892

================ FOLD 4 ================
RMSE=1.2149, MAE=0.8300, R2=0.6361

## XGBoost

In [ ]:
import copy
import json
import os

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

# Path configuration
DATA_DIR = ROOT_DIR / "data" / "input"
CSV_PATH = DATA_DIR / "UAV.csv"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"
SAVE_DIR = DATA_DIR / "UAV" / "XGBoost"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# feature selection
max_feat = max(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # model
    # =========================
    model = xgb.XGBRegressor(
        n_estimators=2000,
        max_depth=1,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=1234
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # =========================
    # prediction
    # =========================
    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"xgb_fold{fold_id}.json")
    model.save_model(model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 165
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 1.0125 RMSE: 1.0063 MAE: 0.7011 R2: 0.6112
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/XGBoost\xgb_fold0.json

================ FOLD 1 ================
Fold 1 | MSE: 1.8793 RMSE: 1.3709 MAE: 1.0839 R2: 0.4495
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/XGBoost\xgb_fold1.json

================ FOLD 2 ================
Fold 2 | MSE: 1.1823 RMSE: 1.0873 MAE: 0.8179 R2: 0.5158
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/XGBoost\xgb_fold2.json

================ FOLD 3 ================
Fold 3 | MSE: 1.4163 RMSE: 1.1901 MAE: 0.8552 R2: 0.5900
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/XGBoost\xgb_fold3.json

================ FOLD 4 ================
Fold 4 | MSE: 1.8378 RMSE: 1.3557 MAE: 0.8845 R2: 0.5469
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/XGBoost\xgb_fold4.json

================ FINAL 5-FOLD RESULT ================
MSE : 1.4657 ± 0.3457


================ FINAL 5-FOLD RESULT ================
MSE : 1.4657 ± 0.3457
RMSE: 1.2020 ± 0.1440
MAE : 0.8685 ± 0.1244
R2  : 0.5427 ± 0.0572

## Random Forest

In [ ]:
import copy
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Path configuration
DATA_DIR = ROOT_DIR / "data" / "input"
CSV_PATH = DATA_DIR / "UAV.csv"
SPLIT_PATH = DATA_DIR / "UAV" / "uav_splits.json"
SAVE_DIR = DATA_DIR / "UAV" / "RandomForest"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# =========================
# feature / target
# =========================
max_feat = max(165, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load splits
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # Random Forest model
    # =========================
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"rf_fold{fold_id}.pkl")
    joblib.dump(model, model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 165
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 1.1607 RMSE: 1.0774 MAE: 0.7202 R2: 0.5543
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/RandomForest\rf_fold0.pkl

================ FOLD 1 ================
Fold 1 | MSE: 1.1730 RMSE: 1.0830 MAE: 0.7213 R2: 0.6564
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/RandomForest\rf_fold1.pkl

================ FOLD 2 ================
Fold 2 | MSE: 0.6251 RMSE: 0.7906 MAE: 0.6049 R2: 0.7440
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/RandomForest\rf_fold2.pkl

================ FOLD 3 ================
Fold 3 | MSE: 1.6247 RMSE: 1.2746 MAE: 0.9116 R2: 0.5297
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/RandomForest\rf_fold3.pkl

================ FOLD 4 ================
Fold 4 | MSE: 1.6107 RMSE: 1.2691 MAE: 0.9446 R2: 0.6029
Saved: E:\LYQ\work\250916remote_model\data\input\UAV/RandomForest\rf_fold4.pkl

================ FINAL 5-FOLD RESULT ================
MSE : 1